In [ ]:
import pandas as pd

In [ ]:
CLASS_NANES = ['World (1)', 'Sports (2)', 'Business (3)', 'Sci/Tech (4)']

ADVERTENCIA!!!


Ejecutar el Jupyter Notebook entero puede ser problemático de cara al consumo de RAM. No se recomienda tratar de ejecutar todo de golpe, es mejor ir por celdas. Las esenciales para que todo funciones son:
- Carga y procesado


Y en caso de aplicar algún clasificador, es necesario cargar df_test.

Cada uno de los Subbloques de clasificadores funcionan de manera independiente, por lo que no es necesario ejecutar todo el bloque de Clasificadores si se desea comprobar alguno en concreto.

## Carga y Preprocesado

In [ ]:
from src.config import PATH_DATA # Ruta para buscar los datos. Por defecto, carpeta ../data/.

from src.preprocesado import cargar_datos
# from src.preprocesado import preprocessing_avanzado, preprocessing_basico, preprocessing_minimal

In [ ]:
df_train = cargar_datos()
df_train.label.value_counts()

## Vectorizaciones

In [ ]:
from src.config import PATH_MODELS # Ruta donde se guardan/leen los modelos.
PATH_MODELS

### Dispersas

In [ ]:
from src.vectorizacion import BoW, TF_IDF

In [ ]:
df_bow, _ = BoW(df_train.desc_avanzado)
df_tfidf, _ = TF_IDF(df_train.desc_basico)

In [ ]:
print("Palabras más comunes en BoW")
print(df_bow.sum(axis=0).sort_values(ascending=False)[:5])
print()
print("Palabras más representativas en TF-IDF")
print(df_tfidf.sum(axis=0).sort_values(ascending=False)[:5])

### Densas

In [ ]:
from src.vectorizacion import get_word2vec_model
from src.vectorizacion import Word2Vec_Embeddings, BERT_Embeddings

In [ ]:
w2v = get_word2vec_model()

In [ ]:
muestra = df_train.head(5)

print("Word2Vec")

df_w2v_muestra = Word2Vec_Embeddings(muestra['desc_basico'], w2v)

print(f"Forma de la matriz: {df_w2v_muestra.shape}") 
display(df_w2v_muestra.head(2)) 


print("BERT")
df_bert_muestra = BERT_Embeddings(muestra['desc_minimal'])

print(f"Forma de la matriz: {df_bert_muestra.shape}")
display(df_bert_muestra.head(2))

## Clasificadores

In [ ]:
df_test = cargar_datos(test=True)
df_test

### Clasificador con BoW + Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
X_train_bow, vectorizador_bow = BoW(df_train['desc_avanzado'], max_features=3000)
X_test_bow,_ = BoW(df_test['desc_avanzado'], max_features=3000,vectorizador=vectorizador_bow)

y_train = df_train['label']
y_test = df_test['label']

In [ ]:
print(f"- Dimensiones Train: {X_train_bow.shape}")
print(f"- Dimensiones Test:  {X_test_bow.shape}")

# Modelo de Naive Bayes
modelo_nb = MultinomialNB()
modelo_nb.fit(X_train_bow, y_train)

# Predicción
y_pred_bow = modelo_nb.predict(X_test_bow)

print("-"*30)
print("- Resultados de Naive Bayes")
print(f"- Accuracy Global: {accuracy_score(y_test, y_pred_bow):.4f}\n")
print("- Desglose por temática:")
print()
print(classification_report(y_test, y_pred_bow, target_names=CLASS_NANES))

### Clasificación con TF-IDF + Regresor Logístico

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
X_train_tfidf, vectorizador_tfidf = TF_IDF(df_train['desc_basico'],max_features=3000)
X_test_tfidf,_ = TF_IDF(df_test['desc_basico'],max_features=3000, vectorizador=vectorizador_tfidf)

y_train = df_train['label']
y_test = df_test['label']

In [ ]:
print(f"- Dimensiones Train: {X_train_tfidf.shape}")
print(f"- Dimensiones Test:  {X_test_tfidf.shape}")

# Modelo de Regresor Logístico
modelo_lr = LogisticRegression(max_iter=1000, random_state=123)
modelo_lr.fit(X_train_tfidf, y_train)

# Predicción
y_pred_tfidf = modelo_lr.predict(X_test_tfidf)

print("-"*30)
print("- Resultados de Regresor Logístico")
print(f"- Accuracy Global: {accuracy_score(y_test, y_pred_tfidf):.4f}\n")
print()
print("- Desglose por temática:")
print(classification_report(y_test, y_pred_tfidf, target_names=CLASS_NANES))

### Clasificación con Word2Vec + RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
w2v = get_word2vec_model()

In [ ]:
X_train_w2v = Word2Vec_Embeddings(df_train['desc_basico'], w2v)
X_test_w2v  = Word2Vec_Embeddings(df_test['desc_basico'], w2v)

y_train = df_train['label']
y_test = df_test['label']

In [ ]:
print(f"- Dimensiones Train: {X_train_tfidf.shape}")
print(f"- Dimensiones Test:  {X_test_tfidf.shape}")

# Modelo de RandomForest
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=123, n_jobs=-1)
modelo_rf.fit(X_train_w2v, y_train)

# Predicción
y_pred_w2v = modelo_rf.predict(X_test_w2v)

print("-"*30)
print("- Resultados de RandomForest")
print(f"- Accuracy Global: {accuracy_score(y_test, y_pred_w2v):.4f}\n")
print()
print("- Desglose por temática:")
print(classification_report(y_test, y_pred_w2v, target_names=CLASS_NANES))

### Clasificador con BERT + MLP

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
ruta_train_bert = PATH_DATA / "train_bert.parquet"
ruta_test_bert  = PATH_DATA / "test_bert.parquet"

y_train = df_train['label']
y_test = df_test['label']

if ruta_train_bert.exists() and ruta_test_bert.exists():
    print("Cargando Embeddings desde almacenamiento...")
    X_train_bert = pd.read_parquet(ruta_train_bert)
    X_test_bert  = pd.read_parquet(ruta_test_bert)
    
else:
    print("Creando Embeddings desde 0...")
    X_train_bert = BERT_Embeddings(df_train['desc_minimal'])
    X_test_bert  = BERT_Embeddings(df_test['desc_minimal'])
    
    X_train_bert.to_parquet(ruta_train_bert, engine="pyarrow", index=False)
    X_test_bert.to_parquet(ruta_test_bert, engine="pyarrow", index=False)


In [ ]:
print(f"- Dimensiones Train: {X_train_bert.shape}")
print(f"- Dimensiones Test:  {X_test_bert.shape}")

modelo_mlp = MLPClassifier(
    hidden_layer_sizes=(128,), 
    activation='relu', 
    solver='adam', 
    early_stopping=True, 
    max_iter=300, 
    random_state=123
)

modelo_mlp.fit(X_train_bert, y_train)

# Predicción
y_pred_bert = modelo_mlp.predict(X_test_bert)

print("-"*30)
print(f"- Accuracy Global: {accuracy_score(y_test, y_pred_bert):.4f}\n")
print()
print("- Desglose por temática:")
print(classification_report(y_test, y_pred_bert, target_names=CLASS_NANES))

## Clasificador con modelo 'Fine-Tuneado'

Existe un script para hacer el fine-tuning que debe de ser ejecutado antes de llegar a esta sección del código.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    DataCollatorWithPadding
)

from src.config import PATH_MODELS

In [ ]:

MODEL_NAME = "distilbert-base-uncased"
ruta_modelo = PATH_MODELS / MODEL_NAME

if not ruta_modelo.exists() or not ruta_modelo.is_dir():
    raise FileNotFoundError(
        f"No se ha encontrado el modelo en '{ruta_modelo}'. "
        "Asegúrate de haber ejecutado el script 'fineTunning.py' y de que la ruta sea correcta."
    )

# Tokenizador y Modelo
print(f"- Cargando tokenizador de {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(ruta_modelo)
modelo_ft = AutoModelForSequenceClassification.from_pretrained(ruta_modelo)

# Se copia para mantener los labels reales en test
df_eval = df_test.copy()
df_eval['label'] = df_eval['label'] - 1 
dataset_test = Dataset.from_pandas(df_eval[['desc_minimal', 'label']])

def tokenize_fn(examples):
    # Misma tokenización que en el script
    return tokenizer(examples["desc_minimal"], truncation=True, max_length=128)

print("- Tokenizando...")
tokenized_test = dataset_test.map(tokenize_fn, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Instanciar Trainer solo para Inferencia (Sin TrainingArguments)
print("- Inferencia...")
trainer = Trainer(
    model=modelo_ft,
    processing_class=tokenizer,
    data_collator=data_collator
)

resultados = trainer.predict(tokenized_test)
y_pred = np.argmax(resultados.predictions, axis=1)

# Volver a hacer la conversión de las labels
y_pred_finetuning = y_pred + 1
y_test_real = df_test['label']


print("-"*30)
print("- Resultados del Fine-Tunning")
print(f"- Accuracy Global: {accuracy_score(y_test, y_pred_finetuning):.4f}\n")
print()
print("- Desglose por temática:")
print(classification_report(y_test_real, y_pred_finetuning, target_names=CLASS_NANES))

## Clasificador con LLM + Propmting

Esquema seguido para los 3 prompts: RTF (Role - Task - Format)

In [ ]:
import re
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
df_muestra = df_test.sample(n=100, random_state=123)[["label","desc_minimal"]].copy()

In [ ]:
llm = ChatOllama(
    model="mixtral:8x7b",
    temperature=0.0
)

def predecir_categoria_langchain(texto_noticia):
    try:
        respuesta_cruda = cadena_clasificacion.invoke({"texto": texto_noticia})
        
        # Por si responde con algo más que el dígito
        match = re.search(r'[1-4]', respuesta_cruda)
        if match:
            return int(match.group(0))
        else:
            return -1

    except Exception as e:
        print(f"Error procesando el texto: {e}")
        return -1


### Zero-Shot

In [ ]:

template = """You are a highly accurate news classification system.
Your task is to classify the following news article into exactly ONE of these four categories:
1 - World
2 - Sports
3 - Business
4 - Sci/Tech

News Article: "{texto}"

Respond ONLY with the single digit (1, 2, 3, or 4). Do not include any explanations, words, or punctuation.
"""

prompt = PromptTemplate.from_template(template)

cadena_clasificacion = prompt | llm | StrOutputParser()


# Predicción
tqdm.pandas(desc="Pensando (Zero-Shot)...")
df_muestra['prediccion_llm'] = df_muestra['desc_minimal'].progress_apply(predecir_categoria_langchain)

# Evaluación
df_validos = df_muestra[df_muestra['prediccion_llm'] != -1]

y_test_llm = df_validos['label']
y_pred_llm = df_validos['prediccion_llm']


print("-"*30)
print(f"- Noticias evaluadas con éxito: {len(df_validos)}/{len(df_muestra)}")
print(f"- Accuracy de la Muestra: {accuracy_score(y_test_llm, y_pred_llm):.4f}\n")
print()
print("- Desglose por temática:")
print(classification_report(y_test_llm, y_pred_llm, target_names=CLASS_NANES, zero_division=0))

### One-Shot

In [ ]:
template_one_shot = """You are a highly accurate news classification system.
Your task is to classify news articles into exactly ONE of these four categories:
1 - World
2 - Sports
3 - Business
4 - Sci/Tech

Respond ONLY with the single digit. Do not include any explanations.

--- Example ---
News Article: "Wall Street surged on Thursday as tech stocks rallied, recovering from earlier losses this week."
Category: 3

--- Now classify this one ---
News Article: "{texto}"
Category: """

prompt = PromptTemplate.from_template(template_one_shot)

cadena_clasificacion = prompt | llm | StrOutputParser()

# Predicción
tqdm.pandas(desc="Pensando (One-Shot)...")
df_muestra['prediccion_llm'] = df_muestra['desc_minimal'].progress_apply(predecir_categoria_langchain)

# Evaluación
df_validos = df_muestra[df_muestra['prediccion_llm'] != -1]

y_test_llm = df_validos['label']
y_pred_llm = df_validos['prediccion_llm']

print("-"*30)
print(f"- Noticias evaluadas con éxito: {len(df_validos)}/{len(df_muestra)}")
print(f"- Accuracy de la Muestra: {accuracy_score(y_test_llm, y_pred_llm):.4f}\n")
print()
print("- Desglose por temática:")
print(classification_report(y_test_llm, y_pred_llm, target_names=CLASS_NANES, zero_division=0))

### Few-Shot

In [ ]:
template_few_shot = """You are a highly accurate news classification system.
Your task is to classify news articles into exactly ONE of these four categories:
1 - World
2 - Sports
3 - Business
4 - Sci/Tech

Respond ONLY with the single digit (1, 2, 3, or 4). Do not include any explanations.

--- Examples ---
News Article: "Peace talks in the Middle East stall after new border skirmishes."
Category: 1

News Article: "Real Madrid secures a stunning victory in the Champions League final with a last-minute goal."
Category: 2

News Article: "Oil prices surge to a six-month high following OPEC's decision to cut global production."
Category: 3

News Article: "Apple announces the new M4 chip, promising unprecedented speeds and AI capabilities for its laptops."
Category: 4

--- Now classify this one ---
News Article: "{texto}"
Category: """

prompt = PromptTemplate.from_template(template_few_shot)

cadena_clasificacion = prompt | llm | StrOutputParser()

# Predicción
tqdm.pandas(desc="Pensando (Few-Shot)...")
df_muestra['prediccion_llm'] = df_muestra['desc_minimal'].progress_apply(predecir_categoria_langchain)

# Evaluación
df_validos = df_muestra[df_muestra['prediccion_llm'] != -1]

y_test_llm = df_validos['label']
y_pred_llm = df_validos['prediccion_llm']

print("-"*30)
print(f"- Noticias evaluadas con éxito: {len(df_validos)}/{len(df_muestra)}")
print(f"- Accuracy de la Muestra: {accuracy_score(y_test_llm, y_pred_llm):.4f}\n")
print()
print("- Desglose por temática:")
print(classification_report(y_test_llm, y_pred_llm, target_names=CLASS_NANES, zero_division=0))